<div class="freebsdLab-IntroCard">
  <div class="freebsdLab-IntroBrand">2. Lab Demo</div>
  <h1 class="freebsdLab-IntroTitle">FreeBSD Laboratory Demonstration</h1>
  <p class="freebsdLab-IntroTagline">Isolation Verification, Network Containment &amp; Runtime Resilience</p>
  <p class="freebsdLab-IntroCopy">This interactive demonstration probes the isolated kernel runtime, validates hypervisor and jail boundaries, audits network isolation across the private laboratory bridge, and measures process stability.</p>
  <div class="freebsdLab-IntroNotice">
    <div class="freebsdLab-IntroNoticeIcon">i</div>
    <div>
      <div class="freebsdLab-IntroNoticeLabel">Laboratory Diagnostics</div>
      <div class="freebsdLab-IntroNoticeText">All commands execute inside disposable, unprivileged runtimes connected via encrypted host-initiated SSH tunnels.</div>
    </div>
    <div class="freebsdLab-IntroNoticeMeta"><strong>FreeBSD</strong>demo + probe</div>
  </div>
</div>

## 1. Runtime Identity & Isolation Properties
Verify hypervisor guest status (`bhyve`), jail isolation, unprivileged process UID, and routing mutation privilege boundaries.

In [1]:
import platform, sys, os, subprocess, socket

def run(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return p.stdout.strip() if p.returncode == 0 else f'ERR ({p.returncode}): {p.stderr.strip()}'

print('=== RUNTIME IDENTITY & ISOLATION PROPERTIES ===')
print('Hypervisor (kern.vm_guest): ', run('sysctl -n kern.vm_guest 2>/dev/null || echo N/A'))
print('Jail ID (security.jail):    ', run('sysctl -n security.jail.jailed 2>/dev/null || echo 0'))
print('Platform:                   ', platform.platform())
print('Runtime Process:            ', f'UID={os.getuid()} (whoami={run("whoami")}), GID={os.getgid()}, PID={os.getpid()}')
print('Route Mutation Privilege:   ', run('route add default 172.31.254.1 2>&1'))
print('Passwordless Sudo:          ', run('sudo -n true 2>&1 || echo Denied / Password Required'))


=== RUNTIME IDENTITY & ISOLATION PROPERTIES ===
Hypervisor (kern.vm_guest):  bhyve
Jail ID (security.jail):     0
Platform:                    FreeBSD-15.1-RELEASE-p2-amd64-64bit-ELF
Runtime Process:             UID=1001 (whoami=freebsd), GID=1001, PID=1222
Route Mutation Privilege:    ERR (77): 
Passwordless Sudo:           sudo: a password is required
Denied / Password Required


## 2. Inbound Management Transport & Loopback Invariant
Audit guest listeners and confirm that Jupyter ZeroMQ channels bind strictly to `127.0.0.1`, carried over the host-initiated SSH tunnel.

In [2]:
print('=== OPEN GUEST LISTENERS (SECURITY AUDIT) ===')
print(run('sockstat -4 -l 2>/dev/null || ss -tulpn 2>/dev/null || netstat -an'))
print()
print('=== ESTABLISHED TRANSPORT CONNECTIONS ===')
print(run('sockstat -4 -c 2>/dev/null || ss -tun 2>/dev/null || netstat -an'))


=== OPEN GUEST LISTENERS (SECURITY AUDIT) ===
USER    COMMAND     PID FD PROTO LOCAL ADDRESS         FOREIGN ADDRESS      
freebsd python3.12 1222  9 tcp4  127.0.0.1:42528       *:*                  
freebsd python3.12 1222 11 tcp4  127.0.0.1:42530       *:*                  
freebsd python3.12 1222 13 tcp4  127.0.0.1:42531       *:*                  
freebsd python3.12 1222 25 tcp4  127.0.0.1:42529       *:*                  
freebsd python3.12 1222 30 tcp4  127.0.0.1:56350       *:*                  
freebsd python3.12 1222 38 tcp4  127.0.0.1:42532       *:*                  
freebsd python3.12 1222 59 tcp4  *:*                   *:*                  
freebsd python3.12 1222 64 udp4  *:*                   *:*                  
freebsd python3.12 1222 65 tcp4  *:*                   *:*                  
root    sshd       1182  7 tcp4  *:22                  *:*                  
root    syslogd     950  7 udp4  *:514                 *:*

=== ESTABLISHED TRANSPORT CONNECTIONS ===
USER 

## 3. Subnet Neighbor & Private Bridge Port Isolation Scan
Probe neighboring addresses across the private laboratory subnet (`172.31.254.10`–`172.31.254.20`) to verify L2 private port isolation.

In [3]:
import socket

def scan_ip_port(ip, port, timeout=0.25):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(timeout)
    try:
        s.connect((ip, port))
        s.close()
        return 'OPEN'
    except socket.timeout:
        return 'FILTERED / TIMED OUT'
    except Exception as e:
        return f'REJECTED ({e})'

print('=== LOCAL SUBNET SCAN (172.31.254.10 - 172.31.254.20) ===')
ifconfig_out = run("ifconfig vtnet0 2>/dev/null | awk '/inet / {print $2}' || ifconfig vnet0 2>/dev/null | awk '/inet / {print $2}' || ip -4 addr show eth0 2>/dev/null | awk '/inet / {print $2}' | cut -d/ -f1")
my_ip = ifconfig_out if ifconfig_out else '172.31.254.10'

for host_num in range(10, 21):
    target = f'172.31.254.{host_num}'
    role = '(Self)' if target == my_ip else '(Neighbor VM)'
    status = scan_ip_port(target, 22)
    print(f'  {target:15s} {role:15s}: SSH/22={status}')

print()
print('=== HOST GATEWAY ACCESS PROBE (172.31.254.1) ===')
print('  Host Gateway SSH (22):   ', scan_ip_port('172.31.254.1', 22))
print('  Host Gateway HTTP (8888):', scan_ip_port('172.31.254.1', 8888))


=== LOCAL SUBNET SCAN (172.31.254.10 - 172.31.254.20) ===


  172.31.254.10   (Neighbor VM)  : SSH/22=FILTERED / TIMED OUT


  172.31.254.11   (Neighbor VM)  : SSH/22=FILTERED / TIMED OUT
  172.31.254.12   (Self)         : SSH/22=OPEN


  172.31.254.13   (Neighbor VM)  : SSH/22=FILTERED / TIMED OUT


  172.31.254.14   (Neighbor VM)  : SSH/22=FILTERED / TIMED OUT


...

  172.31.254.18   (Neighbor VM)  : SSH/22=FILTERED / TIMED OUT


  172.31.254.19   (Neighbor VM)  : SSH/22=FILTERED / TIMED OUT


  172.31.254.20   (Neighbor VM)  : SSH/22=FILTERED / TIMED OUT

=== HOST GATEWAY ACCESS PROBE (172.31.254.1) ===


  Host Gateway SSH (22):    FILTERED / TIMED OUT


  Host Gateway HTTP (8888): FILTERED / TIMED OUT


## 4. Egress Failure Semantics & Routing Audit
Inspect the kernel routing table and demonstrate that external WAN sockets fail immediately at the local route level due to absence of a default gateway.

In [4]:
print('=== ROUTING TABLE INSPECTION ===')
print(run('netstat -rn 2>/dev/null || ip route 2>/dev/null || route -n'))
print()
print('=== DIRECT OUTBOUND SOCKET ATTEMPTS ===')
for ip in ['1.1.1.1', '8.8.8.8']:
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(1.0)
        s.connect((ip, 80))
        print(f'  {ip}:80 -> CONNECTED')
    except OSError as e:
        print(f'  {ip}:80 -> [Errno {e.errno}] {e.strerror}')
        print('         Mechanism: Local guest kernel rejection (no default route).')


=== ROUTING TABLE INSPECTION ===
Routing tables

Internet:
Destination        Gateway            Flags         Netif Expire
127.0.0.1          link#2             UH              lo0
172.31.254.0/24    link#1             U            vtnet0
172.31.254.12      link#2             UHS             lo0

Internet6:
Destination                       Gateway                       Flags         Netif Expire
::/96                             link#2                        URS             lo0
::1                               link#2                        UHS             lo0
::ffff:0.0.0.0/96                 link#2                        URS             lo0
fe80::%lo0/10                     link#2                        URS             lo0
fe80::%lo0/64                     link#2                        U               lo0
fe80::1%lo0                       link#2                        UHS             lo0
ff02::/16                         link#2                        URS             lo0

=== DIRECT

## 5. Computation Durability & Memory Stability
Perform iterative array allocation and math operations to test CPU execution stability and memory persistence inside the runtime.

In [5]:
import time

print('=== COMPUTATION & RUNTIME STABILITY TEST ===')
t0 = time.perf_counter()
data = [i ** 2 for i in range(1_000_000)]
total = sum(data)
elapsed = (time.perf_counter() - t0) * 1000
print(f'Allocated 1M ints, sum = {total}, completed in {elapsed:.2f}ms')
print('Memory allocation and execution pipeline healthy.')


=== COMPUTATION & RUNTIME STABILITY TEST ===
Allocated 1M ints, sum = 333332833333500000, completed in 123.89ms
Memory allocation and execution pipeline healthy.


## 6. Socket Metrics & File Descriptor Leak Check
Inspect the open file descriptors of the kernel process to verify clean socket lifecycle and prevent descriptor leakage.

In [6]:
print('=== FILE DESCRIPTORS & PROCESS METRICS ===')
print('Current Process PID: ', os.getpid())
print('Open File Descriptors:')
print(run(f'procstat -f {os.getpid()} 2>/dev/null | head -n 15 || ls -la /proc/{os.getpid()}/fd 2>/dev/null | head -n 15'))


=== FILE DESCRIPTORS & PROCESS METRICS ===
Current Process PID:  1222
Open File Descriptors:
PID COMM                FD T V FLAGS    REF  OFFSET PRO NAME        
 1222 python3.12        text v r r-------   -       - -   /usr/local/bin/python3.12
 1222 python3.12         cwd v d r-------   -       - -   /home/freebsd     
 1222 python3.12        root v d r-------   -       - -   /                 
 1222 python3.12           0 p - rw------   4       0 -   -                 
 1222 python3.12           1 p - rw------   1       0 -   -                 
 1222 python3.12           2 p - rw------   1       0 -   -                 
 1222 python3.12           3 E - rw---n--   1       0 -   -                 
 1222 python3.12           4 E - rw---n--   2       0 -   -                 
 1222 python3.12           5 k - rw------   2       0 -   -                 
 1222 python3.12           6 E - rw---n--   2       0 -   -                 
 1222 python3.12           7 k - rw------   2       0 -   -  

## 7. Resource Control Enforcement (rctl / ulimits)
Check whether jail resource limits (`rctl`) and standard process ulimits actually cap CPU, process count, open files, and memory for this runtime.

In [7]:
import resource, subprocess

def run(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return p.stdout.strip() if p.returncode == 0 else f'ERR ({p.returncode}): {p.stderr.strip()}'

print('=== RESOURCE CONTROL ENFORCEMENT ===')
print('RCTL usage (rctl -u jail):')
print(run('rctl -u jail 2>/dev/null || echo N/A (rctl unavailable or not jailed)'))
print()
print('Process ulimits:')
limits = [
    ('CPU time (s)', 'RLIMIT_CPU'),
    ('Max processes', 'RLIMIT_NPROC'),
    ('Open files', 'RLIMIT_NOFILE'),
    ('Address space', 'RLIMIT_AS'),
]
for name, attr in limits:
    res = getattr(resource, attr, None)
    if res is None:
        print(f'  {name:20s}: not exposed on this platform')
        continue
    try:
        soft, hard = resource.getrlimit(res)
        print(f'  {name:20s}: soft={soft}, hard={hard}')
    except (ValueError, OSError) as e:
        print(f'  {name:20s}: unavailable ({e})')


=== RESOURCE CONTROL ENFORCEMENT ===
RCTL usage (rctl -u jail):
ERR (2): /bin/sh: Syntax error: "(" unexpected

Process ulimits:
  CPU time (s)        : soft=9223372036854775807, hard=9223372036854775807
  Max processes       : soft=5734, hard=5734
  Open files          : soft=28377, hard=28377
  Address space       : soft=9223372036854775807, hard=9223372036854775807


## 8. Filesystem Root & Device Node Isolation
Audit visible mounts, attempt an unprivileged mount operation, and check for exposure of sensitive device nodes (`/dev/mem`, `/dev/kmem`, raw disks) that should never be reachable from inside an unprivileged jail.

In [8]:
import subprocess

def run(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return p.stdout.strip() if p.returncode == 0 else f'ERR ({p.returncode}): {p.stderr.strip()}'

print('=== VISIBLE MOUNTS ===')
print(run('mount 2>/dev/null || cat /proc/mounts 2>/dev/null'))
print()
print('=== UNPRIVILEGED MOUNT ATTEMPT ===')
print(run('mount -t tmpfs tmpfs /mnt 2>&1 || echo Denied / Not Permitted'))
print()
print('=== SENSITIVE DEVICE NODE EXPOSURE ===')
for dev in ['/dev/mem', '/dev/kmem', '/dev/io', '/dev/da0', '/dev/nvd0', '/dev/vtbd0']:
    print(f'  {dev:15s}: ', run(f'test -e {dev} && ls -la {dev} || echo "not present"'))


=== VISIBLE MOUNTS ===
/dev/gpt/rootfs on / (ufs, local, noatime, soft-updates)
devfs on /dev (devfs)
/dev/gpt/efiboot0 on /boot/efi (msdosfs, local)

=== UNPRIVILEGED MOUNT ATTEMPT ===


mount: tmpfs: Operation not permitted
Denied / Not Permitted

=== SENSITIVE DEVICE NODE EXPOSURE ===
  /dev/mem       :  crw-r-----  1 root kmem 0xf Aug 25 21:43 /dev/mem
  /dev/kmem      :  crw-r-----  1 root kmem 0x10 Aug 25 21:43 /dev/kmem
  /dev/io        :  crw-------  1 root wheel 0x27 Aug 25 21:43 /dev/io


  /dev/da0       :  not present
  /dev/nvd0      :  not present
  /dev/vtbd0     :  crw-r-----  1 root operator 0x59 Aug 25 21:43 /dev/vtbd0


## 9. Kernel Module & Raw Socket Capability Boundary
Attempt to load a kernel module (`kldload`) and open a raw socket — both require elevated capabilities and should be denied for an unprivileged guest UID.

In [9]:
import socket, subprocess

def run(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return p.stdout.strip() if p.returncode == 0 else f'ERR ({p.returncode}): {p.stderr.strip()}'

print('=== KERNEL MODULE LOAD PRIVILEGE ===')
print(run('kldload -n if_tun 2>&1 || echo Denied / Not Permitted'))
print()
print('=== RAW SOCKET CAPABILITY ===')
try:
    rs = socket.socket(socket.AF_INET, socket.SOCK_RAW, socket.IPPROTO_ICMP)
    rs.close()
    print('  Raw ICMP socket: CREATED (elevated network capability present)')
except PermissionError as e:
    print(f'  Raw ICMP socket: DENIED ({e})')
except OSError as e:
    print(f'  Raw ICMP socket: ERROR ({e})')


=== KERNEL MODULE LOAD PRIVILEGE ===
kldload: can't load if_tun: Operation not permitted
Denied / Not Permitted

=== RAW SOCKET CAPABILITY ===
  Raw ICMP socket: DENIED ([Errno 1] Operation not permitted)


## 10. Process & IPC Visibility Scope
Confirm the process table only exposes this jail's own processes (not host daemons or sibling-VM processes), and check for SysV IPC segments that could leak state across tenants.

In [10]:
import subprocess

def run(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return p.stdout.strip() if p.returncode == 0 else f'ERR ({p.returncode}): {p.stderr.strip()}'

print('=== PROCESS TABLE VISIBILITY ===')
print('Visible process count:', run('ps aux 2>/dev/null | wc -l'))
print(run('ps aux 2>/dev/null | head -n 15'))
print()
print('=== SYSV IPC VISIBILITY ===')
print(run('ipcs 2>/dev/null || echo N/A'))


=== PROCESS TABLE VISIBILITY ===
Visible process count: 30
USER     PID %CPU %MEM    VSZ    RSS TT  STAT STARTED     TIME COMMAND
root      11 95.3  0.0      0     16  -  RNL  21:43   42:50.69 [idle]
freebsd 1222  2.3 15.1 190500 152348  -  Ss   21:43    0:04.84 /usr/local/bin/python3 -m ipykernel_launcher -f /tmp/freebsd-laboratory/kernel-79ce8fc2-cf48-4584-b69b-b8973a799730.json (python3.12)
freebsd 1221  0.2  1.1  25792  11604  -  S    21:43    0:00.45 sshd-session: freebsd@notty (sshd-session)
root      12  0.1  0.0      0    448  -  WL   21:43    0:00.28 [intr]
root       0  0.0  0.0      0    464  -  DLs  21:43    0:00.05 [kernel]
root       1  0.0  0.1  12712   1232  -  SLs  21:43    0:00.02 /sbin/init
root       2  0.0  0.0      0     16  -  WL   21:43    0:00.33 [clock]
root       3  0.0  0.0      0     32  -  DL   21:43    0:00.00 [crypto]
root       4  0.0  0.0      0     48  -  DL   21:43    0:00.00 [cam]
root       5  0.0  0.0      0     16  -  DL   21:43    0:00.00 [busdm

## 11. Complementary Egress Vectors (UDP, ICMP, Privileged Bind)
Section 4 only tested TCP egress. Round it out with a UDP connect attempt, an ICMP echo test, and an attempt to bind a privileged (<1024) local port.

In [11]:
import socket, subprocess

def run(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return p.stdout.strip() if p.returncode == 0 else f'ERR ({p.returncode}): {p.stderr.strip()}'

print('=== UDP EGRESS ATTEMPT (DNS-style, 8.8.8.8:53) ===')
try:
    us = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    us.settimeout(1.0)
    us.connect(('8.8.8.8', 53))
    print('  UDP connect() succeeded locally (UDP connect() does not itself confirm reachability)')
    us.close()
except OSError as e:
    print(f'  UDP connect() -> [Errno {e.errno}] {e.strerror}')

print()
print('=== ICMP ECHO (PING) EGRESS ===')
print(run('ping -c 2 -t 2 1.1.1.1 2>&1 || ping -c 2 -W 2 1.1.1.1 2>&1'))

print()
print('=== PRIVILEGED PORT BIND ATTEMPT (tcp/80) ===')
try:
    bs = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    bs.bind(('0.0.0.0', 80))
    bs.close()
    print('  Bind to :80 succeeded (unexpected for an unprivileged UID)')
except PermissionError as e:
    print(f'  Bind to :80 -> DENIED ({e})')
except OSError as e:
    print(f'  Bind to :80 -> ERROR ({e})')


=== UDP EGRESS ATTEMPT (DNS-style, 8.8.8.8:53) ===
  UDP connect() -> [Errno 51] Network is unreachable

=== ICMP ECHO (PING) EGRESS ===


ERR (2): 

=== PRIVILEGED PORT BIND ATTEMPT (tcp/80) ===
  Bind to :80 -> DENIED ([Errno 13] Permission denied)


## 12. Extended Neighbor Surface Scan
Section 3 only probed port 22. Widen the sweep across a small set of common service ports, and inspect the ARP/neighbor table to check for L2-level visibility of sibling VMs even where L3 ports are closed.

In [12]:
import socket, subprocess

def run(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return p.stdout.strip() if p.returncode == 0 else f'ERR ({p.returncode}): {p.stderr.strip()}'

def scan_ip_port(ip, port, timeout=0.25):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(timeout)
    try:
        s.connect((ip, port))
        s.close()
        return 'OPEN'
    except socket.timeout:
        return 'FILTERED / TIMED OUT'
    except Exception as e:
        return f'REJECTED ({e})'

ifconfig_out = run("ifconfig vtnet0 2>/dev/null | awk '/inet / {print $2}' || ifconfig vnet0 2>/dev/null | awk '/inet / {print $2}' || ip -4 addr show eth0 2>/dev/null | awk '/inet / {print $2}' | cut -d/ -f1")
my_ip = ifconfig_out if ifconfig_out else '172.31.254.10'

print('=== EXTENDED NEIGHBOR PORT SCAN (172.31.254.10 - 172.31.254.20) ===')
extra_ports = [22, 80, 443, 445, 3389]
for host_num in range(10, 21):
    target = f'172.31.254.{host_num}'
    if target == my_ip:
        continue
    results = {p: scan_ip_port(target, p, timeout=0.2) for p in extra_ports}
    open_ports = [str(p) for p, s in results.items() if s == 'OPEN']
    print(f'  {target:15s}: open={open_ports if open_ports else "none"}')

print()
print('=== ARP / NEIGHBOR TABLE (L2 VISIBILITY CHECK) ===')
print(run('arp -a 2>/dev/null || ip neigh 2>/dev/null || echo N/A'))


=== EXTENDED NEIGHBOR PORT SCAN (172.31.254.10 - 172.31.254.20) ===


  172.31.254.10  : open=none


  172.31.254.11  : open=none


  172.31.254.13  : open=none


  172.31.254.14  : open=none


...

  172.31.254.16  : open=none


  172.31.254.17  : open=none


  172.31.254.18  : open=none


  172.31.254.19  : open=none


  172.31.254.20  : open=none

=== ARP / NEIGHBOR TABLE (L2 VISIBILITY CHECK) ===
? (172.31.254.13) at (incomplete) on vtnet0 expired [ethernet]
? (172.31.254.12) at 58:9c:fc:0f:9a:37 on vtnet0 permanent [ethernet]
? (172.31.254.15) at (incomplete) on vtnet0 expired [ethernet]
? (172.31.254.14) at (incomplete) on vtnet0 expired [ethernet]
? (172.31.254.11) at (incomplete) on vtnet0 expired [ethernet]
? (172.31.254.10) at (incomplete) on vtnet0 expired [ethernet]
? (172.31.254.1) at 58:9c:fc:10:f2:b2 on vtnet0 expires in 1058 seconds [ethernet]
? (172.31.254.20) at (incomplete) on vtnet0 expired [ethernet]
? (172.31.254.17) at (incomplete) on vtnet0 expired [ethernet]
? (172.31.254.16) at (incomplete) on vtnet0 expired [ethernet]
? (172.31.254.19) at (incomplete) on vtnet0 expired [ethernet]
? (172.31.254.18) at (incomplete) on vtnet0 expired [ethernet]
